In [1]:
import pandas as pd
import zipfile
import urllib.request
import io

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

response = urllib.request.urlopen(url).read()

with zipfile.ZipFile(io.BytesIO(response)) as archive:
    with archive.open("SMSSpamCollection") as file:
        df = pd.read_csv(
            file,
            sep="\t",
            header=None,
            names=["label", "message"]
        )

df = df.drop_duplicates().dropna()

print(df.head())
print("\nDataset size:", len(df))
print("\nMessage categories:")
print(df["label"].value_counts())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset size: 5169

Message categories:
label
ham     4516
spam     653
Name: count, dtype: int64


Prepare the training and testing data

In [3]:
emails = df["message"]
labels = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    emails,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))

Training messages: 4135
Testing messages: 1034


Build the 10-line spam detector

In [4]:
vec = TfidfVectorizer()
model = MultinomialNB()

X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)

model.fit(X_train_vec, y_train)

predictions = model.predict(X_test_vec)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy * 100, "%")

Model Accuracy: 95.26112185686654 %


In [5]:
print("Classification Report:")
print(classification_report(y_test, predictions))

Classification Report:
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       903
        spam       1.00      0.63      0.77       131

    accuracy                           0.95      1034
   macro avg       0.97      0.81      0.87      1034
weighted avg       0.96      0.95      0.95      1034



In [6]:
tests = [
    "Cl@im your fr33 g1ft card now",
    "Not free, just confirming lunch at 1pm",
    "Good afternoon. Kindly verify your account to receive the complimentary reward.",
    "Your parcel is waiting. Pay the small release fee at bit.ly/claim-now",
    "You have won a £500 voucher! Text WIN to 80085 now to claim."
]

expected = [
    "spam",
    "ham",
    "spam",
    "spam",
    "spam"
]

test_predictions = model.predict(vec.transform(tests))

results = pd.DataFrame({
    "My message": tests,
    "Model said": test_predictions,
    "Should be": expected
})

results["Correct?"] = (
    results["Model said"] == results["Should be"]
)

results

,My message,Model said,Should be,Correct?
0,Cl@im your fr33 g1ft card now,ham,spam,False
1,"Not free, just confirming lunch at 1pm",ham,ham,True
2,Good afternoon. Kindly verify your account to ...,ham,spam,False
3,Your parcel is waiting. Pay the small release ...,ham,spam,False
4,You have won a £500 voucher! Text WIN to 80085...,spam,spam,True


In [7]:
reasons = [
    "Spelling tricks hide familiar spam words.",
    "The model may recognize 'free' but miss the negation 'not'.",
    "Polite wording can disguise a suspicious request.",
    "The shortened URL and payment request may be unfamiliar.",
    "Prize and urgency words may or may not match learned spam patterns."
]

results["Why it failed"] = reasons

results

,My message,Model said,Should be,Correct?,Why it failed
0,Cl@im your fr33 g1ft card now,ham,spam,False,Spelling tricks hide familiar spam words.
1,"Not free, just confirming lunch at 1pm",ham,ham,True,The model may recognize 'free' but miss the ne...
2,Good afternoon. Kindly verify your account to ...,ham,spam,False,Polite wording can disguise a suspicious request.
3,Your parcel is waiting. Pay the small release ...,ham,spam,False,The shortened URL and payment request may be u...
4,You have won a £500 voucher! Text WIN to 80085...,spam,spam,True,Prize and urgency words may or may not match l...


In [8]:
print("FINAL SPAM FILTER RESULTS")
print("-------------------------")

print(f"Test Accuracy: {accuracy * 100:.2f}%")

print("\nAdversarial Test Results:")
display(results)

print("\nNumber of incorrect predictions:",
      (results["Correct?"] == False).sum())

FINAL SPAM FILTER RESULTS
-------------------------
Test Accuracy: 95.26%

Adversarial Test Results:


,My message,Model said,Should be,Correct?,Why it failed
0,Cl@im your fr33 g1ft card now,ham,spam,False,Spelling tricks hide familiar spam words.
1,"Not free, just confirming lunch at 1pm",ham,ham,True,The model may recognize 'free' but miss the ne...
2,Good afternoon. Kindly verify your account to ...,ham,spam,False,Polite wording can disguise a suspicious request.
3,Your parcel is waiting. Pay the small release ...,ham,spam,False,The shortened URL and payment request may be u...
4,You have won a £500 voucher! Text WIN to 80085...,spam,spam,True,Prize and urgency words may or may not match l...



Number of incorrect predictions: 3
